<a href="https://colab.research.google.com/github/Anuoluwapo65/pytorch-jax-implementation/blob/main/wanb_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import wandb

In [2]:
wandb.config = {
    "learning_rate": 0.002,
    "batch_size": 200,
    "num_epochs": 125,
    "Architecture": "CNN",
    "Dataset": "CIFAR10",
    "Kernel" : [16, 32],
    "num_classes": 10
 }

In [3]:
import os
import json
import time
import tqdm.notebook as tqdm
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader




In [4]:
def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("using this device", device)

using this device cpu


In [5]:
def get_model(hyperparameters):

  with wandb.init(project = "CIFAR10", config = hyperparameters):

    config = wandb.config

    model, train_loader, test_loader, optimizer, criterion,  = make(config)
    print(model)

    train(model, train_loader, optimizer, criterion, config)

    test(model, test_loader)

  return model

In [6]:
def make(config):
  # Re-initialize datasets and loaders to use config.batch_size
  # Use globally defined train_transform and test_transform
  train_full_dataset = CIFAR10(root = "data_path", download = True, train = True, transform = train_transform)
  test_datasets = CIFAR10(root = "data_path", download = True, train = False, transform = test_transform)

  train_dataset_length = len(train_full_dataset)
  train_set_size = int(0.8 * train_dataset_length)
  val_set_size = train_dataset_length - train_set_size

  train_sets, val_sets = torch.utils.data.random_split(train_full_dataset, [train_set_size, val_set_size])

  train_loader = DataLoader(train_sets, shuffle = True, batch_size = config.batch_size)
  val_loader = DataLoader(val_sets, shuffle = False, batch_size = config.batch_size)
  test_loader = DataLoader(test_datasets, shuffle = False, batch_size = config.batch_size)

  model = ConvNet(config.Kernel, config.num_classes).to(device)

  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr = config.learning_rate)
  return model, train_loader, test_loader, optimizer, criterion

In [7]:
train_datasets = CIFAR10(root = "data_path", download = True)


100%|██████████| 170M/170M [00:04<00:00, 42.6MB/s]


In [8]:
img_mean = ((train_datasets.data)/255.0).mean(axis = (0, 1, 2)) # Corrected axis for per-channel mean
img_std = ((train_datasets.data)/255.0).std(axis =(0, 1, 2)) # Corrected axis for per-channel std
print(img_mean)
print(img_std)

[0.49139968 0.48215841 0.44653091]
[0.24703223 0.24348513 0.26158784]


In [9]:
train_transform = transforms.Compose([
                                transforms.ToTensor(),
                                transforms.Normalize(img_mean, img_std),
                                transforms.RandomHorizontalFlip(p = 0.5),
                                transforms.RandomResizedCrop(32, scale = (0.9, 1.0), ratio = (0.2, 0.8)) # Corrected syntax
                            ])

test_transform = transforms.Compose([
                                     transforms.ToTensor(),
                                     transforms.Normalize(img_mean, img_std) # Corrected argument order
                                    ])


train_dataset_length = len(train_datasets) # Renamed variable to avoid conflict with `train_datasets` object later
train_set_size = int(0.8 * train_dataset_length) # Renamed variable
val_set_size = train_dataset_length - train_set_size # Renamed variable


train_full_dataset = CIFAR10(root = "data_path", download = True, train =True, transform = train_transform) # Renamed variable
test_datasets = CIFAR10(root = "data_path", download = True, train = False, transform = test_transform)

train_sets, val_sets = torch.utils.data.random_split(train_full_dataset, [train_set_size, val_set_size]) # Using renamed variables


train_loader = DataLoader(train_sets, shuffle = True, batch_size = 256) # Pass the subset `train_sets`, not `train_set_size`
val_loader = DataLoader(val_sets, shuffle = False, batch_size = 256) # Pass the subset `val_sets`, not `val_set_size`
test_loader = DataLoader(test_datasets, shuffle = False, batch_size =256)

In [10]:
class ConvNet(nn.Module):
  def __init__(self, kernel, num_classes = 10):
    super(ConvNet, self).__init__()

    self.conv1 = nn.Sequential(
        nn.Conv2d(3, kernel[0], kernel_size = 5,  stride = 2, padding = 1), # Changed in_channels from 1 to 3
        nn.BatchNorm2d(kernel[0]),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2))
    self.conv2 = nn.Sequential(
        nn.Conv2d(kernel[0], kernel[1], kernel_size = 5, stride = 2, padding = 1), # Changed in_channels from 16 to kernel[0]
        nn.BatchNorm2d(kernel[1]),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2)) # Corrected typo: Maxpool2d to MaxPool2d

    # Corrected input features for linear layer based on output of last conv layer
    # Output of last MaxPool2d with current settings is (Batch, kernel[1], 1, 1)
    self.linear1 = nn.Linear(kernel[1] * 1 * 1, num_classes) # Corrected input and output features

  def forward(self, x):
    x = self.conv1(x)
    out = self.conv2(x)
    out = out.reshape(out.size(0), -1)
    out = self.linear1(out)
    return out

In [15]:
def train(model, loader, criterion, optimizer, config):
    # Tell wandb to watch what the model gets up to: gradients, weights, and more!
    wandb.watch(model, criterion, log="all", log_freq=10)

    # Run training and track with wandb
    total_batches = len(loader) * config.num_epochs
    example_ct = 0  # number of examples seen
    batch_ct = 0
    for epoch in tqdm(range(config.num_epochs)):
        for _, (images, labels) in enumerate(loader):

            loss = train_batch(images, labels, model, optimizer, criterion)
            example_ct +=  len(images)
            batch_ct += 1

            # Report metrics every 25th batch
            if ((batch_ct + 1) % 25) == 0:
                train_log(loss, example_ct, epoch)


def train_batch(images, labels, model, optimizer, criterion):
    images, labels = images.to(device), labels.to(device)

    # Forward pass ➡
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass ⬅
    optimizer.zero_grad()
    loss.backward()

    # Step with optimizer
    optimizer.step()

    return loss

In [16]:
def train_log(loss, examples, epoch):

  wandb.log({"epoch":epoch, "loss":loss}, step = examples)
  print(f"Loss after {str(examples).zfill(5)} examples : {loss:.3f}")


In [17]:
def test(model, test_loader):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for label, target in test_loader:
      label = label.to(device)
      target = target.to(device)
      pred = model(label)
      _,predicted = torch.max(pred.data, 1)
      total += target.size(0)
      correct += (predicted == target).sum().item()

    accuracy = correct / total if total > 0 else 0 # Prevent ZeroDivisionError
    print(f"The test accuracy of the model: {100 * accuracy:.2f}%")
    wandb.log({"test_accuracy": accuracy})

  torch.onnx.export(model, label, "model.onnx")
  wandb.save("model.onnx")